# DT401 Week 6 Section 3: Geospatial Data Generalisation

**Module:** Data Fundamentals  
**Week:** 6 of 7  
**Section:** 3 of 3  
**Estimated Time:** 55 minutes

---

## Learning Objectives

Understand how geographic coordinates constitute personal data and apply generalisation techniques to reduce re-identification risk.

By the end of this notebook, you will be able to:
- Explain why GPS coordinates are sensitive personal data under UK GDPR
- Apply coordinate rounding to reduce geographic precision
- Apply random perturbation to displace coordinates by a controlled distance
- Compare rounding and perturbation using displacement distance metrics
- Select the appropriate technique for different business scenarios


---

## How to Use This Notebook

This notebook contains a mix of instructional text, Python code cells, and interactive exercises. Follow the instructions in each section, run the code cells to see outputs, and complete the exercises to reinforce your learning.

You will find several types of boxes throughout the notebook to guide your learning:


<div class="alert alert-block alert-warning">
<b>Reminder boxes:</b> <br>
These boxes contain important reminders about how this section relates to the overall module and your learning journey.
</div>


<div class="alert alert-block alert-info">
<b>Reflection boxes:</b> <br>
These have pointers as to what we should be looking for or comments on how to approach the topic discussed.
</div>


<div class="alert alert-block alert-success">
<b>Task boxes:</b> <br>
These have questions or tasks you should complete before progressing.

These may include hidden hints or solutions.
<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

Hints will guide you toward the solution without giving away the answer completely.

</details>

</div>


---

<div class="alert alert-block alert-warning">
<b>Reminder:</b> <br>
This notebook builds on Section 2 (anonymisation techniques: pseudonymisation, data masking, aggregation, and suppression) by extending those ideas to geospatial data. Location data poses unique privacy risks because even approximate coordinates can identify sensitive locations such as a person's home address, medical appointments, or place of worship. UK GDPR treats location data as personal data when it can identify an individual.
</div>


---

## Part 1: Why Location Data Requires Special Care

### Visual Overview

```mermaid
flowchart LR
    A[Raw GPS Coordinates] --> B[Assess Identification Risk]
    B --> C{Risk Level}
    C -->|High precision| D[Apply Generalisation]
    C -->|Already generalised| E[Use As-Is]
    D --> F[Rounded Coordinates]
    D --> G[Perturbed Coordinates]
    F --> H[Generalised Output for Publication]
    G --> H
```

**What this diagram shows:**  
Raw GPS coordinates carry high precision that can directly identify individuals. Before publishing or sharing location data, you must assess the identification risk and apply an appropriate generalisation technique.

**Why this matters in your role:**  
Any dataset recording where people go - customer visits, service delivery points, workforce locations - contains location data. Without generalisation, even a single GPS point can reveal a person's home address or attendance at a sensitive location.

---

### Coordinate Precision and Privacy

GPS coordinates are recorded as decimal degrees. The number of decimal places determines how precisely a location can be identified:


In [ ]:
import pandas as pd
import numpy as np

precision_table = pd.DataFrame({
    'Decimal Places': [7, 5, 3, 2, 1, 0],
    'Precision (approx)': ['1 cm', '1 m', '111 m', '1.1 km', '11 km', '111 km'],
    'Can identify': [
        'Exact location on street',
        'Individual building',
        'City block',
        'Neighbourhood',
        'Town area',
        'County/Region'
    ]
})

print(precision_table.to_string(index=False))


<div class="alert alert-block alert-info">
<b>Why precision matters:</b> <br>
At 3 decimal places, a coordinate is accurate to roughly 111 metres - enough to identify someone's home to a city block. At 2 decimal places, precision drops to approximately 1.1 km, placing someone only within a neighbourhood. When publishing location data, choosing the right number of decimal places is a deliberate privacy decision, not just a formatting choice.
</div>


---

## Part 2: Coordinate Rounding

### Visual Overview

```mermaid
flowchart LR
    A["Original Coordinates<br/>51.5080, -0.1281"] --> B[Round to N Decimal Places]
    B --> C["Reduced Precision Point<br/>51.508, -0.128"]
    C --> D[Publish or Share]
```

**What this diagram shows:**  
Rounding truncates the coordinate to a set number of decimal places, moving the reported location to the nearest grid point at that precision level.

---

### How Coordinate Rounding Works

Coordinate rounding is the simplest generalisation technique. You simply reduce the number of decimal places in the latitude and longitude values. This:

- Is **irreversible** - the original precise location cannot be recovered from the rounded value
- Is **deterministic** - the same input always produces the same output
- Moves the point to the nearest grid intersection at that precision level
- Applies uniformly across an entire dataset

The example below uses Trafalgar Square (51.5080, -0.1281), a well-known public landmark, for demonstration purposes.


<div class="alert alert-block alert-warning">
<b>Reminder:</b> <br>
Rounding is deterministic - the same coordinate always rounds to the same value. This consistency is useful for joining datasets but means a determined attacker who knows your rounding level can narrow down the original location to a grid square. Perturbation (Part 3) addresses this by introducing randomness.
</div>


In [ ]:
import plotly.graph_objects as go

# Original coordinates (Trafalgar Square, London - a public landmark)
latitude = 51.5080
longitude = -0.1281

# Round to 3 decimal places (~111m precision)
decimal_places = 3
lat_rounded = round(latitude, decimal_places)
lon_rounded = round(longitude, decimal_places)

print(f"Original:  {latitude}, {longitude}")
print(f"Rounded:   {lat_rounded}, {lon_rounded}")
print(f"Decimal places used: {decimal_places}")

# Visualise both points on an interactive map
fig = go.Figure()
fig.add_trace(go.Scattermapbox(
    lat=[latitude], lon=[longitude],
    mode='markers',
    marker=dict(size=12, color='blue'),
    name='Original'
))
fig.add_trace(go.Scattermapbox(
    lat=[lat_rounded], lon=[lon_rounded],
    mode='markers',
    marker=dict(size=12, color='green'),
    name=f'Rounded ({decimal_places} dp)'
))
fig.update_layout(
    mapbox=dict(style='carto-positron', zoom=14,
        center=dict(lat=latitude, lon=longitude)),
    height=400, legend=dict(x=0, y=1),
    margin=dict(l=0, r=0, t=30, b=0)
)
fig.show()


<div class="alert alert-block alert-info">
<b>Reading the map:</b> <br>
The blue point shows the original coordinate. The green point shows the rounded coordinate. At 3 decimal places the displacement is small but visible at street level - the green point snaps to the nearest 111-metre grid intersection. Zoom in to see the gap between the two points. The rounded location still conveys the general area but hides the precise position.
</div>


<div class="alert alert-block alert-success">
<b>Task 2.1:</b> <br>
Change <code>decimal_places</code> to <code>2</code> in the code above and re-run the cell. Observe how the displacement between the original and rounded points changes. Then answer: at 2 decimal places, what neighbourhood-level area does the rounded coordinate represent?

<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

Set <code>decimal_places = 2</code> and run the cell again. The green point will jump to a coarser grid. Use the precision table from Part 1 to identify the approximate area size.

</details>

<details>
    <summary style="color:green;font-weight:bold">Click here for the solution</summary>

Set <code>decimal_places = 2</code>. The rounded coordinates become <code>51.51, -0.13</code>. At 2 decimal places, precision is approximately 1.1 km, so the point now represents a neighbourhood rather than a city block. The original location is concealed within a roughly 1 km square area.

</details>

</div>


In [ ]:
# Task 2.1 - change decimal_places to 2 and observe the difference
# You can copy the code from the cell above or modify decimal_places directly there

# Your code here:


---

## Part 3: Coordinate Perturbation

### Visual Overview

```mermaid
flowchart LR
    A[Original Coordinates] --> B[Generate Random Offset]
    B --> C[Add Noise Within Radius]
    C --> D[Perturbed Point]
    D --> E[Publish or Share]
    F["Perturbation Radius e.g. 200m"] --> B
```

**What this diagram shows:**  
Perturbation adds a random spatial offset to the original coordinate. The offset is bounded by a specified radius, so the reported location lands somewhere within a circle around the true position.

---

### How Coordinate Perturbation Works

Coordinate perturbation adds random noise to latitude and longitude values. Unlike rounding, the displacement is random within a defined radius:

- The maximum displacement is controlled by the perturbation radius (in metres)
- **Non-deterministic** - the same input produces a different output each time the code runs
- Harder to reverse than rounding because the attacker does not know the direction or magnitude of the offset
- Well-suited to scenarios where you want maximum unpredictability

The conversion from metres to degrees accounts for the fact that one degree of longitude covers a different physical distance at different latitudes.


<div class="alert alert-block alert-warning">
<b>Reminder:</b> <br>
Because perturbation is non-deterministic, running the cell multiple times will produce a different perturbed location each time. In this example, <code>np.random.seed(42)</code> is used to fix the random number generator so results are reproducible for demonstration. In a real anonymisation pipeline, you would typically remove the seed so that each run produces a genuinely different offset.
</div>


In [ ]:
from geopy.distance import geodesic

# Perturbation parameters
perturbation_radius_m = 200  # displacement radius in metres

# Convert metres to approximate degrees
# 1 degree latitude  approx 111,320 metres
# 1 degree longitude approx 111,320 * cos(latitude) metres
lat_range = perturbation_radius_m / 111320
lon_range = perturbation_radius_m / (111320 * np.cos(np.radians(latitude)))

np.random.seed(42)  # For reproducibility in this example
lat_perturbed = latitude + np.random.uniform(-lat_range, lat_range)
lon_perturbed = longitude + np.random.uniform(-lon_range, lon_range)

# Calculate actual displacement distance
original_point = (latitude, longitude)
perturbed_point = (lat_perturbed, lon_perturbed)
displacement_m = geodesic(original_point, perturbed_point).meters

print(f"Original:    {latitude}, {longitude}")
print(f"Perturbed:   {lat_perturbed:.7f}, {lon_perturbed:.7f}")
print(f"Displacement: {displacement_m:.1f} metres (max allowed: {perturbation_radius_m}m)")

# Visualise
fig2 = go.Figure()
fig2.add_trace(go.Scattermapbox(
    lat=[latitude], lon=[longitude],
    mode='markers',
    marker=dict(size=12, color='blue'),
    name='Original'
))
fig2.add_trace(go.Scattermapbox(
    lat=[lat_perturbed], lon=[lon_perturbed],
    mode='markers',
    marker=dict(size=12, color='red'),
    name=f'Perturbed (+-{perturbation_radius_m}m)'
))
fig2.update_layout(
    mapbox=dict(style='carto-positron', zoom=14,
        center=dict(lat=latitude, lon=longitude)),
    height=400, legend=dict(x=0, y=1),
    margin=dict(l=0, r=0, t=30, b=0)
)
fig2.show()


<div class="alert alert-block alert-info">
<b>Reading the map:</b> <br>
The blue point is the original coordinate. The red point is the perturbed coordinate. Notice that the red point could land anywhere within a 200-metre circle around the original. The printed displacement confirms the actual distance moved. Unlike rounding, the perturbed point does not snap to a regular grid - its position is genuinely unpredictable to anyone who does not have the random seed.
</div>


<div class="alert alert-block alert-success">
<b>Task 3.1:</b> <br>
Change <code>perturbation_radius_m</code> to <code>500</code> and re-run the cell. Observe how the displacement increases. Then remove the <code>np.random.seed(42)</code> line and run the cell three times. Note whether the perturbed location changes each run.

<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

Set <code>perturbation_radius_m = 500</code> and run the cell. The displacement should now be up to 500 metres. Removing the seed line means the random offset is different each run - this is the non-deterministic behaviour described above.

</details>

<details>
    <summary style="color:green;font-weight:bold">Click here for the solution</summary>

With <code>perturbation_radius_m = 500</code>, the maximum displacement increases to 500 metres and the perturbed point can land anywhere within a 500-metre circle. Without <code>np.random.seed(42)</code>, each run of the cell generates a fresh random offset, so the red point moves to a different location every time. This illustrates the non-deterministic nature of perturbation.

</details>

</div>


In [ ]:
# Task 3.1 - change perturbation_radius_m to 500 and observe
# Also try removing np.random.seed(42) and running multiple times

# Your code here:


---

## Part 4: Comparing Techniques

### Visual Overview

```mermaid
flowchart TD
    A[Original Coordinates] --> B[Rounding]
    A --> C[Perturbation]
    B --> D["Fixed grid point<br/>Deterministic displacement"]
    C --> E["Random point within radius<br/>Variable displacement"]
    D --> F[Comparison]
    E --> F
    F --> G{"Which technique fits<br/>the use case?"}
    G -->|Consistent joins needed| H[Use Rounding]
    G -->|Maximum unpredictability| I[Use Perturbation]
```

**What this diagram shows:**  
Both techniques displace the original location, but in fundamentally different ways. The right choice depends on whether you need consistency across records or unpredictability.

---

### Comparing Rounding and Perturbation Side by Side


In [ ]:
# Calculate displacement for rounding
round_displacement = geodesic((latitude, longitude), (lat_rounded, lon_rounded)).meters

# Comparison table
comparison = pd.DataFrame({
    'Technique': ['Rounding (3 dp)', 'Perturbation (200m)'],
    'Displacement (m)': [round(round_displacement, 1), round(displacement_m, 1)],
    'Deterministic': ['Yes - same output every run', 'No - different output each run'],
    'Best for': [
        'Consistent anonymisation across datasets',
        'Maximum unpredictability'
    ]
})

print(comparison.to_string(index=False))

# Combined map showing all three points
fig3 = go.Figure()
fig3.add_trace(go.Scattermapbox(
    lat=[latitude], lon=[longitude],
    mode='markers',
    marker=dict(size=14, color='blue'),
    name='Original'
))
fig3.add_trace(go.Scattermapbox(
    lat=[lat_rounded], lon=[lon_rounded],
    mode='markers',
    marker=dict(size=12, color='green'),
    name='Rounded (3 dp)'
))
fig3.add_trace(go.Scattermapbox(
    lat=[lat_perturbed], lon=[lon_perturbed],
    mode='markers',
    marker=dict(size=12, color='red'),
    name='Perturbed (200m)'
))
fig3.update_layout(
    mapbox=dict(style='carto-positron', zoom=14,
        center=dict(lat=latitude, lon=longitude)),
    height=450, legend=dict(x=0, y=1),
    margin=dict(l=0, r=0, t=30, b=0)
)
fig3.show()


<div class="alert alert-block alert-info">
<b>Comparing the techniques:</b> <br>
The combined map shows all three points. Blue is the original. Green (rounded) always lands at the same grid intersection regardless of how many times you run the notebook. Red (perturbed) lands at a random position within the 200-metre circle each run (when no seed is set). Rounding produces consistent, auditable results - useful when two teams need to independently anonymise the same dataset and get matching outputs. Perturbation produces unpredictable results - useful when you want to make reverse-engineering harder.
</div>


<div class="alert alert-block alert-success">
<b>Task 4.1 - Scenario:</b> <br>
An NHS trust wants to share de-identified patient visit location data with a university research team to analyse health service demand patterns across the city. The research team will join this location data with publicly available census data using grid squares.

Which generalisation technique - rounding or perturbation - is more appropriate here, and why?

<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

Consider whether the research team needs locations to be consistent across records (e.g., two visits to the same area always land in the same grid square) or whether unpredictability is the priority. Also consider whether the NHS trust and the research team need to independently arrive at the same anonymised coordinates.

</details>

<details>
    <summary style="color:green;font-weight:bold">Click here for the solution</summary>

Rounding is more appropriate for this scenario. The research team needs to join location data to census grid squares, which requires consistent grid alignment. Rounding ensures that all visits within the same 111-metre grid square are assigned the same rounded coordinate, making spatial joins reliable. Perturbation would assign different coordinates to visits at the same location, breaking the join logic. The NHS trust and research team can also independently apply the same rounding rule and arrive at identical anonymised datasets, which supports audit and reproducibility.

</details>

</div>


*Your answer here...*

**Recommended technique:**

**Reasoning:**


---

## Part 5: Independent Task - Anonymising a Set of Locations

### Scenario

A local council wants to publish the locations of social care visits in order to understand how services are distributed across the borough. Publishing exact coordinates would reveal individual home addresses, which is prohibited under UK GDPR. The council's data governance team has approved a generalisation approach using either rounding or perturbation.

Your task is to apply **both** techniques to a set of five sample coordinates and present the results so the data governance team can evaluate the options.

---

### Sample Data

The five coordinates below represent public locations in the London borough area used as stand-ins for actual visit addresses:

| ID | Description | Latitude | Longitude |
|----|-------------|----------|-----------|
| 1 | Victoria Station area | 51.4952 | -0.1441 |
| 2 | Lambeth Town Hall area | 51.4613 | -0.1156 |
| 3 | Brixton Market area | 51.4627 | -0.1145 |
| 4 | Kennington Park area | 51.4840 | -0.1072 |
| 5 | Elephant and Castle area | 51.4958 | -0.1005 |

### Your Task

1. Load the coordinates into a pandas DataFrame
2. Apply rounding to 3 decimal places to produce a  and  column
3. Apply perturbation with a radius of 100 metres to produce a  and  column
4. Calculate the displacement in metres for both techniques for each point
5. Display the full results DataFrame
6. Plot all original, rounded, and perturbed points on a single interactive map


<div class="alert alert-block alert-success">
<b>Task 5.1:</b> <br>
Complete the steps described above in the code cell below. Use the techniques demonstrated in Parts 2 and 3. Display your results as a DataFrame and on an interactive map.

<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

Structure your work in stages:

1. Create a DataFrame with the five coordinates
2. Use <code>df["lat_rounded"] = df["lat_original"].round(3)</code> to apply rounding
3. For perturbation, calculate <code>lat_range</code> and <code>lon_range</code> for each row using the formula from Part 3, then add a random offset
4. Use <code>geodesic()</code> in a loop to calculate displacement for each row
5. For the map, add one trace per technique with all latitudes listed together

</details>

<details>
    <summary style="color:green;font-weight:bold">Click here for the solution</summary>

```python
import pandas as pd
import numpy as np
from geopy.distance import geodesic
import plotly.graph_objects as go

# Sample data
locations = pd.DataFrame({
    "ID": [1, 2, 3, 4, 5],
    "Description": [
        "Victoria Station area",
        "Lambeth Town Hall area",
        "Brixton Market area",
        "Kennington Park area",
        "Elephant and Castle area"
    ],
    "lat_original": [51.4952, 51.4613, 51.4627, 51.4840, 51.4958],
    "lon_original": [-0.1441, -0.1156, -0.1145, -0.1072, -0.1005]
})

# Apply rounding (3 decimal places)
locations["lat_rounded"] = locations["lat_original"].round(3)
locations["lon_rounded"] = locations["lon_original"].round(3)

# Apply perturbation (100m radius)
radius_m = 100
np.random.seed(7)
lat_offsets = [np.random.uniform(-radius_m / 111320, radius_m / 111320) for _ in range(len(locations))]
lon_offsets = [
    np.random.uniform(
        -radius_m / (111320 * np.cos(np.radians(lat))),
         radius_m / (111320 * np.cos(np.radians(lat)))
    )
    for lat in locations["lat_original"]
]
locations["lat_perturbed"] = locations["lat_original"] + lat_offsets
locations["lon_perturbed"] = locations["lon_original"] + lon_offsets

# Calculate displacements
locations["round_displacement_m"] = [
    round(geodesic((row.lat_original, row.lon_original), (row.lat_rounded, row.lon_rounded)).meters, 1)
    for row in locations.itertuples()
]
locations["perturb_displacement_m"] = [
    round(geodesic((row.lat_original, row.lon_original), (row.lat_perturbed, row.lon_perturbed)).meters, 1)
    for row in locations.itertuples()
]

print(locations[["ID", "Description", "lat_original", "lon_original",
                  "lat_rounded", "lon_rounded", "round_displacement_m",
                  "lat_perturbed", "lon_perturbed", "perturb_displacement_m"]].to_string(index=False))

# Map
fig_task = go.Figure()
fig_task.add_trace(go.Scattermapbox(
    lat=locations["lat_original"], lon=locations["lon_original"],
    mode="markers", marker=dict(size=12, color="blue"), name="Original"
))
fig_task.add_trace(go.Scattermapbox(
    lat=locations["lat_rounded"], lon=locations["lon_rounded"],
    mode="markers", marker=dict(size=10, color="green"), name="Rounded (3 dp)"
))
fig_task.add_trace(go.Scattermapbox(
    lat=locations["lat_perturbed"], lon=locations["lon_perturbed"],
    mode="markers", marker=dict(size=10, color="red"), name="Perturbed (100m)"
))
fig_task.update_layout(
    mapbox=dict(style="carto-positron", zoom=12, center=dict(lat=51.478, lon=-0.122)),
    height=500, legend=dict(x=0, y=1), margin=dict(l=0, r=0, t=30, b=0)
)
fig_task.show()
```

</details>

</div>


In [ ]:
# Task 5.1 - Anonymise the five council locations using both techniques

# Your code here:


<div class="alert alert-block alert-info">
<b>Success criteria:</b> <br>
Your completed task should produce:

- A DataFrame with columns for original, rounded, and perturbed coordinates, plus displacement distances for each technique
- An interactive map with three sets of markers (blue = original, green = rounded, red = perturbed) for all five locations
- Displacement values showing rounding displacements up to approximately 78 metres (half the 111-metre grid diagonal) and perturbation displacements up to 100 metres
</div>


---

## Reflection

Reflect on the following question and record your thoughts in the cell below. Ground your answer in an example from your workplace, using **approximately 150 words** in total.

Your answer here forms part of your Module Challenge Part A portfolio and is assessed against **MLO3:** *Select and apply appropriate data-gathering methods for a given data analysis scenario*.

Where you draw on external sources, cite them in Harvard style. This does not count towards your word count.

Once you commit your notebook, your Coach will be able to provide feedback on your response. You can develop your answer further before your final submission.

<div class="alert alert-block alert-info">

1. Coordinate rounding is a deterministic technique; perturbation adds random noise. If you needed to run a further spatial analysis on the resulting data afterwards, which technique would preserve more analytical value, and what would you lose either way?

</div>

*Write your responses here.*

1. 

References (if applicable):

---

## Summary

**You have completed Week 6 Section 3.** You can now:

- Explain why GPS coordinates are sensitive personal data under UK GDPR, and how the number of decimal places in a coordinate determines identification risk
- Apply coordinate rounding with Python's `round()` to reduce geographic precision to a chosen level, such as 3 decimal places for city-block precision
- Apply random perturbation with `np.random.uniform()` to displace coordinates by a controlled random distance within a specified radius
- Calculate and compare displacement distances for both techniques using geopy's `geodesic()` function
- Select rounding when consistent, auditable anonymisation is needed, and perturbation when maximum unpredictability is required
- Apply both techniques to a dataset of five sample locations and present the results in a DataFrame and on an interactive Plotly map

**Key takeaways:**

- Converting a perturbation radius from metres to degrees must account for latitude: longitude degrees cover less physical distance at higher latitudes, so the longitude offset divides by `cos(latitude)` while the latitude offset does not.
- Rounding's displacement is bounded by the precision grid, up to roughly 78 metres at 3 decimal places (half the diagonal of a 111-metre cell), while perturbation's displacement is bounded directly by the chosen radius.

**What comes next:**

- **Week 7 Section 1, "Working Through a Data Analysis"** brings together the data quality, gathering, and analysis skills from across the module into an end-to-end workflow, preparing you for the Module Challenge.

**Action items:**

1. Save this notebook (Ctrl+S or Cmd+S)
2. Commit and push your work to your GitHub repository with the commit message `"Complete Week 6 Section 3 geospatial data generalisation"`
3. Make sure the repository shows your latest commit and that all files are up to date
4. Share a link to your repository with your Coach in the DT401.6 component on Aptem
